In [0]:
master_df = spark.table("workspace.default.employee_incremental_week_7")

display(master_df)

employee_id,employee_name,work_location,email_address,department,employment_status
E103,Rohan Gupta,Mumbai,rohan.gupta@company.com,Finance,Active
E107,Rahul Mehta,Hyderabad,rahul.mehta@company.com,Engineering,Active
E112,Meera Joshi,Lucknow,meera.joshi@company.com,Support,Active
E118,Riya Sen,Patna,riya.sen@company.com,Sales,Active
E120,Tanvi Mishra,Delhi,tanvi.mishra@company.com,Marketing,Active
E121,Aditi Khanna,Gurugram,aditi.khanna@company.com,Engineering,Active
E122,Mohit Arora,Pune,mohit.arora@company.com,Finance,Active
E123,Sakshi Bansal,Chennai,sakshi.bansal@company.com,HR,Active
E124,Dev Agarwal,Jaipur,dev.agarwal@company.com,Support,Active
E125,Ankit Tiwari,Bengaluru,ankit.tiwari@company.com,Sales,Active


In [0]:
incremental_df = spark.table("workspace.default.employee_master_week_7")

display(incremental_df)

employee_id,employee_name,work_location,email_address,department,employment_status
E101,Aarav Sharma,Bengaluru,aarav.sharma@company.com,Engineering,Active
E102,Diya Verma,Hyderabad,diya.verma@company.com,HR,Active
E103,Rohan Gupta,Pune,rohan.gupta@company.com,Finance,Inactive
E104,Sneha Iyer,Chennai,sneha.iyer@company.com,Marketing,Active
E105,Kabir Singh,Delhi,kabir.singh@company.com,Sales,Active
E106,Ananya Das,Kolkata,ananya.das@company.com,Support,Active
E107,Rahul Mehta,Mumbai,rahul.mehta@company.com,Engineering,Active
E108,Neha Jain,Jaipur,neha.jain@company.com,Finance,Inactive
E109,Arjun Rao,Ahmedabad,arjun.rao@company.com,Engineering,Active
E110,Priya Nair,Kochi,priya.nair@company.com,HR,Active


In [0]:
master_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.employee_delta")

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "workspace.default.employee_delta")

deltaTable.alias("target") \
.merge(
    incremental_df.alias("source"),
    "target.employee_id = source.employee_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
result = spark.table("workspace.default.employee_delta")

display(result)

employee_id,employee_name,work_location,email_address,department,employment_status
E121,Aditi Khanna,Gurugram,aditi.khanna@company.com,Engineering,Active
E122,Mohit Arora,Pune,mohit.arora@company.com,Finance,Active
E123,Sakshi Bansal,Chennai,sakshi.bansal@company.com,HR,Active
E124,Dev Agarwal,Jaipur,dev.agarwal@company.com,Support,Active
E125,Ankit Tiwari,Bengaluru,ankit.tiwari@company.com,Sales,Active
E101,Aarav Sharma,Bengaluru,aarav.sharma@company.com,Engineering,Active
E102,Diya Verma,Hyderabad,diya.verma@company.com,HR,Active
E103,Rohan Gupta,Pune,rohan.gupta@company.com,Finance,Inactive
E104,Sneha Iyer,Chennai,sneha.iyer@company.com,Marketing,Active
E105,Kabir Singh,Delhi,kabir.singh@company.com,Sales,Active


In [0]:
result.createOrReplaceTempView("employees")

In [0]:
%sql
SELECT *
FROM employees
ORDER BY employee_id;

employee_id,employee_name,work_location,email_address,department,employment_status
E101,Aarav Sharma,Bengaluru,aarav.sharma@company.com,Engineering,Active
E102,Diya Verma,Hyderabad,diya.verma@company.com,HR,Active
E103,Rohan Gupta,Pune,rohan.gupta@company.com,Finance,Inactive
E104,Sneha Iyer,Chennai,sneha.iyer@company.com,Marketing,Active
E105,Kabir Singh,Delhi,kabir.singh@company.com,Sales,Active
E106,Ananya Das,Kolkata,ananya.das@company.com,Support,Active
E107,Rahul Mehta,Mumbai,rahul.mehta@company.com,Engineering,Active
E108,Neha Jain,Jaipur,neha.jain@company.com,Finance,Inactive
E109,Arjun Rao,Ahmedabad,arjun.rao@company.com,Engineering,Active
E110,Priya Nair,Kochi,priya.nair@company.com,HR,Active


In [0]:
%sql
SELECT *
FROM employees
WHERE employee_id IN
('E103','E107','E112','E118','E120');

employee_id,employee_name,work_location,email_address,department,employment_status
E103,Rohan Gupta,Pune,rohan.gupta@company.com,Finance,Inactive
E107,Rahul Mehta,Mumbai,rahul.mehta@company.com,Engineering,Active
E112,Meera Joshi,Lucknow,meera.joshi@company.com,Support,Inactive
E118,Riya Sen,Patna,riya.sen@company.com,Sales,Inactive
E120,Tanvi Mishra,Varanasi,tanvi.mishra@company.com,Marketing,Active


In [0]:
%sql
SELECT *
FROM employees
WHERE employee_id >= 'E121';

employee_id,employee_name,work_location,email_address,department,employment_status
E121,Aditi Khanna,Gurugram,aditi.khanna@company.com,Engineering,Active
E122,Mohit Arora,Pune,mohit.arora@company.com,Finance,Active
E123,Sakshi Bansal,Chennai,sakshi.bansal@company.com,HR,Active
E124,Dev Agarwal,Jaipur,dev.agarwal@company.com,Support,Active
E125,Ankit Tiwari,Bengaluru,ankit.tiwari@company.com,Sales,Active


In [0]:
display(
    deltaTable.history()
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-08-02T19:49:17.000Z,75629475753231,harshkashyap1221@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1929783837813308),edfa2469-6d9e-476f-81c2-b9bbe8d8cae6,0802-194127-85ca89x5-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 5952, p25FileSize -> 3070, numDeletionVectorsRemoved -> 1, minFileSize -> 3070, numAddedFiles -> 1, maxFileSize -> 3070, p75FileSize -> 3070, p50FileSize -> 3070, numAddedBytes -> 3070)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-08-02T19:49:14.000Z,75629475753231,harshkashyap1221@gmail.com,MERGE,"Map(predicate -> [""(employee_id#12947 = employee_id#12959)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1929783837813308),edfa2469-6d9e-476f-81c2-b9bbe8d8cae6,0802-194127-85ca89x5-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2882, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 20, executionTimeMs -> 3193, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1268, numTargetRowsUpdated -> 20, numOutputRows -> 20, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 20, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1891)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-02T19:49:00.000Z,75629475753231,harshkashyap1221@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1929783837813308),0ae7adb7-5b2f-4912-86ec-728eb66f8ed4,0802-194127-85ca89x5-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 5952, p25FileSize -> 3070, numDeletionVectorsRemoved -> 1, minFileSize -> 3070, numAddedFiles -> 1, maxFileSize -> 3070, p75FileSize -> 3070, p50FileSize -> 3070, numAddedBytes -> 3070)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-02T19:48:58.000Z,75629475753231,harshkashyap1221@gmail.com,MERGE,"Map(predicate -> [""(employee_id#12371 = employee_id#12383)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1929783837813308),0ae7adb7-5b2f-4912-86ec-728eb66f8ed4,0802-194127-85ca89x5-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2882, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 20, executionTimeMs -> 3195, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1353, numTargetRowsUpdated -> 20, numOutputRows -> 20, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 20, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1808)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-02T19:48:47.000Z,75629475753231,harshkashyap1221@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1929783837813308),32525f9a-7c3d-4d55-bb0d-8ba70ba67e51,0802-194127-85ca89x5-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 5524, p25FileSize -> 3070, nu

In [0]:
spark.read.format("delta") \
.option("versionAsOf",0) \
.load("/tmp/employee_delta")

In [0]:
spark.sql("""
VACUUM workspace.default.employee_delta
RETAIN 168 HOURS
""")

DataFrame[path: string]